# Implement your own federated recommender

This notebook defines a new recommendation method and runs it on a small synthetic table. The method is a federated popularity baseline. Each client counts the items in its training rows and uploads that vector once. The server adds the vectors with the existing `Sum` aggregator. Scoring ranks items by that sum.

The client and the server are ordinary subclasses. They use the same round loop, the same server configuration and the same evaluator as the built-in methods. The classes live in this notebook.

`examples/quickstart.ipynb` runs FedNCF on MovieLens-100K. `examples/custom_data.ipynb` shows how to load a table you already have. The table below is a smaller copy of that synthetic table, so this run finishes immediately.


## The popularity model

The score of an item is the number of training interactions that name it. Items that appear more often score higher. The model is that count vector: every user is scored with the same numbers. The list a user is shown still depends on the user, because the evaluator removes items from the excluded splits before it ranks. Validation candidates drop that user's training items. Test candidates drop training and validation items.

The server stores the vector under the name `item_counts`, one entry per item. Item ids are the contiguous ids produced by `InteractionDataset`, so entry 0 is item 0. The vector starts at zero. The server only ever sees the uploaded vectors.

The first time a client is selected, it counts item ids in its own training rows and uploads a vector of the same length. Items that client never saw stay at zero. `Sum` adds the uploaded vectors. The group is exchanged as a `delta`, so the server computes `item_counts = item_counts + that sum`. After every client has uploaded once, `item_counts[j]` is the number of training rows for item `j`.

A client that is selected again sends no `item_counts` package. Those counts are already in the vector, and another upload would add them a second time. One algorithm object is shared by every client, so the "already uploaded" flag is stored in the client state the server keeps for that client.

`delta` is the exchange that lets a client selected in a later round add its counts to whatever is already stored. With `exchange="params"` the aggregated upload would become the new vector, and counts from earlier rounds would be replaced by the clients selected in the current round.


## Two classes and one `ParamIO`

A method is a client algorithm plus a server.

`PopularityClientAlgorithm` subclasses `BaseClientAlgorithm`.

- `initialize_local_parameters` returns the upload flag, set to false. The server calls it once per client while it builds client state.
- `local_train` reads `training_request.training_data`, builds the count vector, and puts it in `updates_packages` under the name `item_counts`. The request also carries `validation_data`. Those rows stay unread, so held-out interactions never enter the counts.
- `compute_scores` reads `global_parameters["item_counts"]` and returns one score per candidate item, for each user on that client. The evaluator calls this hook. It does not call `local_train`, and it does not pass labels.

`PopularityServer` subclasses `BaseFederatedServer`. The method it implements is `_initialize_server_parameters`, which returns the zero vector. Selecting clients, building the payload, calling `local_train`, aggregating and applying the result stay in the base class. `run` is that loop.

The exchanged group is declared with `ParamIO`. The client copies that object from the training request through `_update_package`.

| Field | Value | What the round loop does with it |
| --- | --- | --- |
| `residency` | `"server"` | The vector lives in `global_parameters`, and the payload sent to a client includes it. |
| `exchange` | `"delta"` | The aggregated upload is added to the current vector. |
| `apply_to` | `"server"` | The result is written into `global_parameters`. |
| `layout` | `"dense"` | Each upload is the whole vector, including zeros. |

`layout="rows"` is the other layout. Embedding methods use it to upload `{"rows", "values"}` for the item rows they touched. `Sum` treats a missing row as zero, so a dense count vector and a row upload of the positive counts are the same sum. This notebook uploads the dense vector, which is a plain array.

The server class accepts whichever aggregator you pass for the group. It does not inspect the aggregator's class. `Sum` is the aggregator this model needs: the global count is the sum of the local counts, and `Sum` only needs that sum. Pass a different aggregator and the same uploads are combined by that aggregator's rule instead.

The base server's default `training_framework` is `"torch"`. Before aggregation it converts uploads to tensors on `server.device`. The zero vector is therefore created on `self.device`, and `delta` can add the aggregated tensor to it. The histogram itself is built with NumPy. A non-finite count raises. The library leaves non-finite values as failures.


## Install

FedRecLab is installed from GitHub. The command also installs NumPy, PyTorch, pandas, tqdm and Cornac. PyTorch is a large download. Run this cell before any import. The notebook does not use the repository working directory, so the same cell works in Colab from a fresh environment.


In [ ]:
%pip install "git+https://github.com/aymenkhs/FedRecLab.git"


## Load a small dataset

The frame is drawn in memory: 40 users, 60 items, 12 distinct items per user, ratings from 1 to 5, and increasing timestamps. User ids start at 1001 and item ids at 50001. `load_dataframe_dataset` remaps both to contiguous integers starting at 0, so 0 is a real user and a real item after this step. `user_id_map` and `item_id_map` translate the original ids.

The split is stratified by user and chronological, as in the quickstart: each user's earlier interactions stay in training, 10% go to validation and 20% to test. With 12 rows per user that holdout fits. Every user has training rows, which is what the counts are computed from.

`CrossDevicePartitionStrategy` assigns each training user to their own client. `seed=42` is stored on the dataset and on the split.

`examples/custom_data.ipynb` covers filters, foreign column names and a downloaded table. This cell only needs a dataset the server can train on.


In [ ]:
import numpy as np
import pandas as pd

from fedreclab.data.client_partitioning import CrossDevicePartitionStrategy
from fedreclab.data.load_data import load_dataframe_dataset
from fedreclab.data.split import StratifiedSplitStrategy

SEED = 42
rng = np.random.default_rng(SEED)

n_users = 40
n_items = 60
per_user = 12

user_ids = np.repeat(np.arange(1001, 1001 + n_users), per_user)
item_pool = np.arange(50_001, 50_001 + n_items)
item_ids = np.concatenate(
    [rng.choice(item_pool, size=per_user, replace=False) for _ in range(n_users)]
)
ratings = rng.integers(1, 6, size=user_ids.shape[0]).astype(float)
base = rng.integers(1_600_000_000, 1_700_000_000, size=n_users)
offsets = np.sort(rng.integers(0, 10_000_000, size=(n_users, per_user)), axis=1)
timestamps = (base[:, None] + offsets).ravel()

interactions = pd.DataFrame(
    {
        "user_id": user_ids,
        "item_id": item_ids,
        "rating": ratings,
        "timestamp": timestamps,
    }
)

dataset = load_dataframe_dataset(
    interactions,
    splitting_strategy=StratifiedSplitStrategy(
        include_validation=True,
        include_test=True,
        val_size=0.1,
        test_size=0.2,
        chrono=True,
        by="user",
        seed=SEED,
    ),
    client_partitioning_strategy=CrossDevicePartitionStrategy(),
    include_validation=True,
    include_test=True,
    seed=SEED,
)

print(
    f"{dataset.num_users} users, {dataset.num_items} items, "
    f"{dataset.num_clients} clients"
)
print("original user 1001 ->", int(dataset.user_id_map[1001]))
print("original item 50001 ->", int(dataset.item_id_map[50001]))
print(dataset.interaction_data["split"].value_counts())


## The client

`local_train` runs once per selected client per round. On the first selection it builds a count for every catalogue item with `numpy.bincount`. Item id 0 is included, because 0 is a real id. The length of the vector is the length of `item_counts` in the server payload, which is how the client learns the catalogue size the server is holding.

The package name must be `item_counts`, the `ParamIO` group. `_update_package` attaches the `ParamIO` from the request. The returned state sets `uploaded` to true. A later round for the same client returns an empty `updates_packages`: the base server skips a group that nobody sent, and the vector is left as it is.

`training_losses` is an empty list. There is no optimisation step, so there is no loss to report. `num_training_examples` is still the number of training rows. `Sum` ignores that number. An aggregator such as `FedAvg` would weight by it, which is why the response reports it anyway.

`compute_scores` moves the server tensor to a NumPy vector and indexes it with each user's candidate ids. The same vector is used for every user. Higher is a more frequent training item.

Methods that learn a network also implement `init_model_skeleton`, which builds the shared `torch.nn.Module`. Popularity has no module, so this class leaves that hook unimplemented. `apply_privacy` is the hook that runs after `local_train` and before aggregation. The base implementation returns the upload unchanged, which is what this method wants.


In [ ]:
import time

import numpy as np

from fedreclab.federated.base_client import (
    BaseClientAlgorithm,
    ClientStateBase,
    ClientTrainingRequest,
    ClientTrainingResponse,
)


class PopularityClientAlgorithm(BaseClientAlgorithm):
    """Upload each client's training item counts once, then score by the sum.

    One instance is shared by every client. The upload flag lives in the
    client state the server stores for that client.

    Simulation shortcuts:
        None. Counts come from the training rows on the request.
    """

    def initialize_local_parameters(
        self,
        client_id,
        total_num_users,
        num_users,
        num_items,
        client_train_data_view,
        users_in_client,
    ):
        """Return the per-client upload flag, initially false.

        Parameters
        ----------
        client_id, total_num_users, num_users, num_items,
        client_train_data_view, users_in_client
            Signature required by the server. The count is computed later
            from the training request, so every client starts from the
            same flag.

        Returns
        -------
        dict
            ``uploaded`` false.
        """
        return {"uploaded": False}

    def local_train(self, training_request: ClientTrainingRequest) -> ClientTrainingResponse:
        """Upload a dense count vector the first time this client is selected.

        Parameters
        ----------
        training_request : ClientTrainingRequest
            Training rows, server payload and the ParamIO map copied into
            the upload.

        Returns
        -------
        ClientTrainingResponse
            An ``item_counts`` package on the first call for this client.
            Later calls send no package. ``training_losses`` is empty.

        Raises
        ------
        ValueError
            If a count is non-finite.
        """
        t0 = time.perf_counter()
        state = training_request.client_state
        training_data = training_request.training_data
        uploaded = bool(state.local_parameters["uploaded"])

        updates = {}
        if not uploaded:
            num_items = int(training_request.global_payload["item_counts"].shape[0])
            item_ids = training_data["item_id"].to_numpy(dtype=np.int64)
            # First selection only. A second upload would be added again.
            counts = np.bincount(item_ids, minlength=num_items).astype(np.float32)
            if not np.isfinite(counts).all():
                raise ValueError(
                    f"Client {training_request.client_id} produced non-finite item counts"
                )
            updates["item_counts"] = self._update_package(
                training_request, "item_counts", counts
            )
            uploaded = True

        updated_state = ClientStateBase(
            client_id=training_request.client_id,
            local_parameters={"uploaded": uploaded},
        )
        return ClientTrainingResponse(
            client_id=training_request.client_id,
            updated_client_state=updated_state,
            updates_packages=updates,
            training_losses=[],
            training_duration=time.perf_counter() - t0,
            num_training_examples=int(len(training_data)),
        )

    def compute_scores(
        self,
        client_id,
        candidate_items_per_user,
        global_parameters,
        client_state,
        extra_payload,
    ):
        """Score candidates by the server's item counts.

        Parameters
        ----------
        client_id : int
            Client being scored. The vector does not depend on it.
        candidate_items_per_user : dict of int to ndarray
            Item ids to score, keyed by user id.
        global_parameters : dict
            Server tensor ``item_counts``.
        client_state : ClientStateBase
            Present for the evaluator signature. Popularity keeps no
            user-specific parameters.
        extra_payload : dict
            Present for the evaluator signature.

        Returns
        -------
        dict of int to ndarray
            One score array per user, aligned with that user's candidates.
            A higher score is a more frequent training item.
        """
        counts = global_parameters["item_counts"].detach().cpu().numpy()
        return {
            user_id: counts[np.asarray(items, dtype=np.int64)]
            for user_id, items in candidate_items_per_user.items()
        }


## The server

The constructor declares the single group and forwards everything else to `BaseFederatedServer`: the dataset, the client algorithm, the number of rounds, the seed, the selector, the aggregators and the evaluator. Those are the same arguments a built-in server takes, plus `param_ios`, which this class fills in.

`_initialize_server_parameters` returns a float32 zero tensor on `self.device`. Float32 holds these counts exactly. The device matches the tensors the base server will add during the `delta` update.


In [ ]:
import torch

from fedreclab.federated.base_server import BaseFederatedServer
from fedreclab.federated.utils import ParamIO


class PopularityServer(BaseFederatedServer):
    """Server for the federated popularity baseline.

    The only override is ``_initialize_server_parameters``. Selection,
    the round loop and aggregation stay on the base class. Pass ``Sum``
    for the ``item_counts`` group.

    Simulation shortcuts:
        None. The server stores the aggregated uploads and does not
        read the interaction table.
    """

    def __init__(self, **kwargs):
        """Declare ``item_counts`` and forward the experiment configuration.

        Parameters
        ----------
        **kwargs
            Arguments of ``BaseFederatedServer`` other than ``param_ios``,
            which this class sets: the dataset, the client algorithm,
            ``num_rounds``, the selector, the aggregators and the evaluator.
        """
        param_ios = {
            "item_counts": ParamIO(
                residency="server",
                exchange="delta",
                apply_to="server",
                layout="dense",
            ),
        }
        super().__init__(param_ios=param_ios, **kwargs)

    def _initialize_server_parameters(self):
        """Return a zero count for every remapped item.

        Returns
        -------
        dict
            ``item_counts``, shape ``(num_items,)``, float32, on ``self.device``.
        """
        num_items = self.interaction_dataset.num_items
        # On self.device so delta can add the aggregated tensor, which the
        # base server has already moved to that device.
        return {
            "item_counts": torch.zeros(
                num_items, dtype=torch.float32, device=self.device
            )
        }


## Configure the run

Three objects, built for this server only, then the server itself.

`FullClientSelection` selects every client in every round. `num_rounds=2`. Round 0 receives a count vector from every client, and the `Sum` of those vectors is added to the zeros. Round 1 selects the same clients again. Each of them finds `uploaded` set and sends no package, so `item_counts` stays at the training counts. The next section checks that equality against a direct count of the training split. That check reads the whole training table. The method never does: each client saw only its own training view.

`FullRankingEvaluator` is the evaluator the built-in examples use. It averages NDCG and hit-rate at 10. Scoring calls `compute_scores` on the client algorithm passed in, so the evaluator and the server share that algorithm.

`aggregation_strategies` maps the group name to one `Sum`. `num_clients` on `Sum` is the population size stored by the aggregator base class. `seed=42` seeds PyTorch on the server. This method draws no random numbers of its own.

Client algorithms, selectors, aggregators and evaluators are single-use. Build new ones for every server. The dataset may be shared.

`early_stopping` stays at its default, off. Once every client has uploaded, further rounds do not change the vector, so there is no best-round snapshot to recover. The test score below uses `global_parameters` and `client_states` from the end of `run`. The FedNCF quickstart turns early stopping on and scores `best_global_parameters`, `best_client_states` and `best_extra_payload` instead, because that training run can peak and then worsen.


In [ ]:
from fedreclab.data.evaluator import FullRankingEvaluator
from fedreclab.federated.aggregation import Sum
from fedreclab.federated.client_selection import FullClientSelection

client = PopularityClientAlgorithm()
evaluator = FullRankingEvaluator(
    metrics=["ndcg", "hr"],
    topk=10,
    interaction_dataset=dataset,
    client_algorithm=client,
)
server = PopularityServer(
    interaction_dataset=dataset,
    client_algorithm=client,
    num_rounds=2,
    seed=SEED,
    client_selection_strategy=FullClientSelection(dataset.num_clients),
    aggregation_strategies={"item_counts": Sum(num_clients=dataset.num_clients)},
    evaluator=evaluator,
)


## Train

`run` executes both rounds. An evaluator is attached, so each round ends with a full-catalogue validation. On this table that pass is small. `current_metrics` is the validation result from the last round that was scored.


In [ ]:
server.run()
print(server.current_metrics)


## Check the count vector

Count the training split directly and compare it to `item_counts`. The two vectors match, and their sums equal the number of training rows, after two rounds in which every client was selected twice. Every client state has `uploaded` set. The five largest entries are the items the model will rank first, before the evaluator drops items a given user has already seen.


In [ ]:
train_counts = np.bincount(
    dataset.train_interactions["item_id"].to_numpy(dtype=np.int64),
    minlength=dataset.num_items,
).astype(np.float64)
server_counts = (
    server.global_parameters["item_counts"].detach().cpu().numpy().astype(np.float64)
)
uploaded = sum(
    1
    for state in server.client_states.values()
    if state.local_parameters["uploaded"]
)

print(
    f"training rows {int(train_counts.sum())}, "
    f"sum of item_counts {float(server_counts.sum())}"
)
print(f"clients that uploaded: {uploaded} / {dataset.num_clients}")
if uploaded != dataset.num_clients:
    raise AssertionError("a client finished the run without uploading")
if not np.array_equal(server_counts, train_counts):
    raise AssertionError(
        "item_counts does not match a direct count of the training split"
    )
print("item_counts matches the training split")

print("highest training counts (remapped item id, count):")
for item_id in np.argsort(-server_counts)[:5]:
    print(f"  {int(item_id)}: {int(server_counts[item_id])}")


## Evaluate

Score the test split with the parameters `run` finished with. The result is a dict with keys `ndcg@10` and `hr@10`. Each value is the mean over users who have test interactions.

These are full-ranking numbers for this table and this popularity model. A learned method is compared by running it on the same dataset and scoring it with `FullRankingEvaluator` at the same cut-off.


In [ ]:
print(evaluator.evaluate(
    current_round=server.num_rounds - 1,
    global_parameters=server.global_parameters,
    client_states=server.client_states,
    extra_payload=server.extra_payload,
    split="test",
))


## Read the built-in methods

The methods shipped with the library are the same two classes, with a real local update in place of the histogram. In a clone of the repository each one is a folder under `src/fedreclab/methods/<name>/`. After installation the same modules are `fedreclab.methods.<name>`.

A folder contains:

- `<name>.py`, the server, a `BaseFederatedServer`
- `<name>_client.py`, the client algorithm
- `<name>_model.py`, the network, when the method has one

The class docstrings state the task, the loss, what is exchanged, which aggregator matches the paper, and any simulation shortcut.

Start with FedMF. It is the smallest learned method, and it also uses `Sum`. The server holds an item-factor table. Each client holds that user's factors, trains on the user's ratings, and uploads an item gradient. The group uses `exchange="gradient"` and `layout="rows"`. The server still only implements `_initialize_server_parameters`. Read `fedmf.py` and `fedmf_client.py`.

FedNCF is the method in the quickstart. Three groups are declared. The quickstart passes `SimpleAvg` for the item embeddings and `FedAvg` for the shared network. The server subclass is again only the initial parameters.

The popularity server leaves several hooks at their defaults. The following methods are where those hooks are used:

- FedVAE applies an aggregated gradient with a server-side Adam step in `_apply_aggregated_result_custom`.
- FedPerGNN builds neighbour state in `_before_training`, `_post_selection_callback` and `_on_round_end_server`. The client implements `apply_privacy`.
- FedFast reclusters in `_on_round_end_server`.
- GPFedRec and SemiDFEGL build a client-specific payload in `_build_server_payload`. MetaMF does that and applies the aggregated meta-network gradient in `_apply_aggregated_result_custom`. EFVAE advances its KL-anneal counter in `_on_round_end_server`. SemiDFEGL also implements `apply_privacy` on the client.

The loop those classes sit in is `BaseFederatedServer.run` and `run_round` in `src/fedreclab/federated/base_server.py`. The request, the response and `ClientStateBase` are in `base_client.py`. `ParamIO` is in `federated/utils.py`. `Sum` and the other aggregators are in `federated/aggregation.py`.

The cell below prints the FedMF files in the environment you just installed, which is the copy to open next.


In [ ]:
import fedreclab.methods.fedmf.fedmf as fedmf_server
import fedreclab.methods.fedmf.fedmf_client as fedmf_client

print(fedmf_server.__file__)
print(fedmf_client.__file__)
